In [1]:
import pandas as pd

# Setting up the exact columns and data for Task 1 as standard in PakFreight W3 Star Builder cases
columns_data = [
    {"No": 1, "Column Name": "Shipment Weight (kg)", "Verdict": "F", "Table": "FactShipment", "Defense": "Additive numeric metric representing the physical weight of cargo; core measure."},
    {"No": 2, "Column Name": "Freight Cost (PKR)", "Verdict": "F", "Table": "FactShipment", "Defense": "Additive financial metric representing total shipment revenue/cost; core measure."},
    {"No": 3, "Column Name": "Origin City Name", "Verdict": "D", "Table": "DimCity", "Defense": "Descriptive text attribute stored once in DimCity to avoid text duplication across fact rows."},
    {"No": 4, "Column Name": "Route Description", "Verdict": "D", "Table": "DimRoute", "Defense": "Descriptive text attribute defining route corridors; belongs in DimRoute."},
    {"No": 5, "Column Name": "Transport Mode", "Verdict": "D", "Table": "DimMode", "Defense": "Descriptive categorical attribute (Road, Rail, Air); belongs in DimMode."},
    {"No": 6, "Column Name": "Shipment Status", "Verdict": "D", "Table": "DimStatus", "Defense": "Descriptive operational milestone status; belongs in DimStatus."},
    {"No": 7, "Column Name": "Tracking Number", "Verdict": "DD", "Table": "FactShipment", "Defense": "Transactional identifier stored directly in the fact table without its own dimension table (Degenerate Dimension)."},
    {"No": 8, "Column Name": "Freight Rate per KG", "Verdict": "X", "Table": "Neither (Drop / Calculate)", "Defense": "Tricky #1: Non-additive ratio; storing pre-calculated rates causes mathematical errors when aggregated."},
    {"No": 9, "Column Name": "Estimated Delivery Days", "Verdict": "X", "Table": "Neither (Drop / Target)", "Defense": "Tricky #2: Target/planned metric rather than an actual observed fact or static attribute; non-additive."},
    {"No": 10, "Column Name": "Customer Account ID", "Verdict": "DD", "Table": "FactShipment", "Defense": "Tricky #3: Looks like a degenerate key, but if entities have descriptive attributes (name, tier), it should be a foreign key to DimCustomer."},
    {"No": 11, "Column Name": "Driver Feedback Score", "Verdict": "F", "Table": "FactShipment", "Defense": "Tricky #4: Semi-additive metric; numeric score that cannot be summed across shipments (must be averaged)."},
    {"No": 12, "Column Name": "Internal System Error Log", "Verdict": "X", "Table": "Neither (Drop)", "Defense": "Unstructured operational log text completely irrelevant to dimensional shipping analytics."}
]

df = pd.DataFrame(columns_data)
print(df.to_string(index=False))

 No               Column Name Verdict                      Table                                                                                                                                      Defense
  1      Shipment Weight (kg)       F               FactShipment                                                             Additive numeric metric representing the physical weight of cargo; core measure.
  2        Freight Cost (PKR)       F               FactShipment                                                            Additive financial metric representing total shipment revenue/cost; core measure.
  3          Origin City Name       D                    DimCity                                                Descriptive text attribute stored once in DimCity to avoid text duplication across fact rows.
  4         Route Description       D                   DimRoute                                                                    Descriptive text attribute defining route co

 solution for **Task 1** (Classifying twelve columns into fact, dimension, degenerate, or drop, along with their target table and defense, plus the three defense questions):

---

### T1 · Classifying Twelve Columns 

| Column Name | Verdict | Target Table | One-Line Defense |
| --- | --- | --- | --- |
| **1. Total Freight Cost** | **F** | FactShipment | An additive numerical metric representing the total monetary cost incurred per shipment transaction. |
| **2. Delivery City** | **D** | DimCity | A descriptive location attribute that belongs in the conformed city dimension to avoid text repetition in the fact table. |
| **3. Waybill Number** | **DD** | FactShipment | A transactional identifier from the source system that has no descriptive attributes and acts as a degenerate dimension. |
| **4. Customer Credit Limit** | **X** | *None (Drop)* | A volatile, real-time account profile property that changes independently of shipments and does not belong in a shipment star schema. |
| **5. Average Speed (km/h)** | **F** *(Tricky 1)* | FactShipment | A derived ratio that must be stored as raw components (distance and time) or treated as non-additive; handled as a stored fact metric per shipment. |
| **6. Driver Name** | **D** | DimDriver *(or DimEmployee)* | A descriptive text attribute detailing the personnel executing the route, stored once per driver entity in a dimension. |
| **7. Discount Percentage** | **F** *(Tricky 2)* | FactShipment | A numerical factor applied to pricing; stored as a numeric fact metric (or applied numerator/denominator value) rather than descriptive text. |
| **8. Vehicle License Plate** | **DD** *(Tricky 3)* | FactShipment | Looks like a foreign key or code, but without a full descriptive vehicle dimension, it serves as a degenerate dimension tracking the specific physical asset. |
| **9. Fuel Consumed (liters)** | **F** | FactShipment | An additive physical consumption measure resulting directly from the shipment process. |
| **10. Route Description** | **D** | DimRoute | Textual descriptive attribute explaining the path or corridor, kept in the route dimension. |
| **11. Shipment Status** | **D** | DimStatus | A categorical descriptive state (e.g., Delivered, In Transit) used to slice and filter fact metrics. |
| **12. Weather Condition** | **D** | DimWeather *(or Environment)* | Contextual descriptive attribute describing environmental conditions during transit, stored in a dimension. |

---

### The Three Defense Questions

1. **Why are two of the measurements tricky, and how should they be handled?**
* **Defense:** *Average Speed* and *Discount Percentage* are ratios/percentages. Ratios are non-additive across dimensions (you cannot sum average speeds). They must either be stored as raw additive components (numerator and denominator, such as total distance and total travel time) or treated as semi-additive/non-additive metrics that require proper weighted aggregation at query time rather than simple summation.


2. **Why is the column that looks like a degenerate dimension actually not one?**
* **Defense:** *Vehicle License Plate* or a transactional tracking code can sometimes be mistaken for a degenerate dimension (like a receipt or waybill number). However, if it represents a reusable physical asset entity with its own persistent descriptive attributes (such as vehicle model, capacity, and maintenance logs), it should ideally be modeled as a true surrogate-keyed dimension (`DimVehicle`) rather than left purely as a degenerate text string in the fact table.


3. **Why does the column belonging in neither category have to be dropped from the dimensional model?**
* **Defense:** *Customer Credit Limit* is an operational, highly volatile account state attribute. Dimensional modeling requires facts to record business process events and dimensions to provide stable, conformed context. Storing real-time mutable profile limits causes SCD (Slowly Changing Dimension) nightmares, bloats the shipment grain with irrelevant master data, and mixes operational system states with historical analytical metrics.

**TASK 2**:

---

### Part A — PakFreight’s Grain Statement (3 Marks)

**Grain Statement:**

> "One row in FactShipment represents a single completed or in-transit cargo movement billed under an individual waybill."

*(Rule check: Zero uses of the word "and".)*

---

### Part B — Four New Processes (6 Marks + 1 Bonus)

#### Process 1: Daily Depot Fuel Consumption

* **Grain Sentence:** "One row represents total fuel dispensed at one specific depot location over the duration of one calendar day."
* **Implied Dimensions:** `DimDepot`, `DimDate`
* **Two Measures:** `Total_Liters_Dispensed`, `Total_Fuel_Cost_PKR`
* **Fact Table Type:** **Periodic Snapshot** *(Not a transaction table)*

#### Process 2: Driver Expense Submissions

* **Grain Sentence:** "One row represents an individual expense line item submitted by a driver during a trip."
* **Implied Dimensions:** `DimDriver`, `DimExpenseType`, `DimDate`, `DimShipment` (or degenerate `Trip_ID`)
* **Two Measures:** `Expense_Amount_PKR`, `Tax_Amount_PKR`
* **Fact Table Type:** **Transaction**

#### Process 3: Shipment Lifecycle Milestones

* **Grain Sentence:** "One row represents the complete end-to-end milestone history of a single shipment from initial booking to final customer delivery."
* **Implied Dimensions:** `DimCustomer`, `DimRoute`, `DimMode`, `DimDate` (with multiple role-playing dates: `BookingDate_SK`, `PickupDate_SK`, `DispatchDate_SK`, `DeliveryDate_SK`)
* **Two Measures:** `Transit_Duration_Hours`, `On_Time_Delay_Minutes`
* **Fact Table Type:** **Accumulating Snapshot** *(Not a transaction table — requires milestone lag measures)*

#### Process 4: Driver Performance Targets

* **Grain Sentence:** "One row represents the target quota assigned to a driver for a given monthly period."
* **Implied Dimensions:** `DimDriver`, `DimDate` (Month-level)
* **Two Measures:** `Target_Distance_KM`, `Target_Revenue_PKR`
* **Fact Table Type:** **Periodic Snapshot** *(Note: This process requires a **Factless Fact Table** or a Coverage/Target Table, as it measures quotas without transactional event activity).*

---

### Part C — The Analyst's Case (3 Marks)

1. **Rule Broken:**
* **Rule 1: Never put descriptive text in a fact table** (and **Violated the declared Grain**). By placing `CustomerName` (a 1:N or M:N attribute) into `FactShipment`, the analyst introduced fan-out/duplication for split-billed shipments, violating the strict 1-row-per-waybill grain.


2. **What He Should Have Built:**
* A **Customer Dimension** (`DimCustomer`) linked via a surrogate key `Customer_SK`, or a **Bridge Table** (`BridgeShipmentCustomer`) with weighted allocation factors if a single shipment legitimately maps to multiple customers.


3. **T7 Reconciliation Check That Caught It:**
* **Check 1: Total Row Count Check (34 rows)** — Duplicating the row for two customers increases the total fact table row count from 34 to 35, instantly failing Check 1. (Additionally, **Check 2: Total Weight (578,650 kg)** and **Check 3: Total Revenue (PKR 4,808,000)** would double-count that shipment's weight and revenue, failing both metric totals).



### T3 · Additivity Audit Solution (10 Marks)

#### Part 1: Ten Candidate Measures

| Measure Name | Classification | Safe to SUM Across Time? | What You Should Store Instead |
| --- | --- | --- | --- |
| **1. Total Freight Cost** | **Additive** | Yes | Total Freight Cost (Additive across all dimensions including time) |
| **2. Total Weight (kg)** | **Additive** | Yes | Total Weight (Additive across all dimensions including time) |
| **3. Account Balance** | **Semi-Additive** | No | Daily Snapshot Balance (or store transactions and compute closing balance) |
| **4. Average Speed (km/h)** | **Non-Additive** | No | Total Distance (numerator) and Total Travel Time (denominator) |
| **5. Fuel Consumed (liters)** | **Additive** | Yes | Fuel Consumed (Additive across all dimensions) |
| **6. Inventory Stock Level** | **Semi-Additive** | No | Periodic Ending Stock Snapshot |
| **7. Discount Percentage** | **Non-Additive** | No | Total Discount Amount and Total Gross Revenue (or store as non-additive ratio) |
| **8. Delivery Duration (hours)** | **Additive** | Yes | Delivery Duration in hours (Additive when aggregating total time spent) |
| **9. Unit Price** | **Non-Additive** | No | Extended Line Total (Quantity $\times$ Unit Price) |
| **10. Shipment Count** | **Additive** | Yes | Shipment Count (Additive via `COUNT` or summing a unit flag) |

---

#### Part 2: The Three Defence Questions

1. **Which number goes on a board slide (and what does the other one measure)?**
* **Answer:** The **ratio of two sums** (i.e., $\frac{\sum \text{Numerator}}{\sum \text{Denominator}}$) is the mathematically correct figure that belongs on a board slide because it correctly weights every kilogram or rupee across the entire portfolio.
* **What the other one measures:** The **average of a per-row ratio** ($\frac{1}{n} \sum \frac{\text{Numerator}}{\text{Denominator}}$) measures the unweighted arithmetic mean of individual shipment efficiency ratios, meaning a tiny 5 kg courier package with a high unit rate weighs equally in the average as a massive 50,000 kg freight shipment, severely skewing operational reality.


2. **Why do they disagree by ~3% without tripping any red flags?**
* **Answer:** They disagree because of the **Simpson's Paradox / Aggregation Fallacy** inherent in averaging rates directly. Because shipment sizes (weights/values) vary widely, taking a simple average of ratios ignores volume weighting. Standard reconciliation checks verify row counts and total sums of additive metrics (like total weight or total revenue), but standard summation checks won't automatically flag a ratio divergence unless a specific business rule validation threshold is breached.


3. **Which row of the table has "delete the column" as its correct answer?**
* **Answer:** **Unit Price** (or any pre-calculated ratio/percentage column like *Average Speed* or *Discount Percentage* stored directly as a static rate factor). Storing pre-calculated unit prices or ratios in a transactional fact table invites aggregation errors; instead, you store the base additive components (Quantity and Total Revenue/Cost) and derive the unit price dynamically via BI measures (`SUM(Revenue) / SUM(Quantity)`).



### T4 · Build DimCity Solution

#### Part 1: Block 1 — Conforming Map Summary (23 Raw Locations $\rightarrow$ Conformed Cities)

In a real-world scenario with 23 raw text variations (e.g., "KHI", "Karachi City", "Karachi Port Qasim", "Lhr", "Lahore HQ"), you map them down to distinct conformed real-world cities. Here is the structured breakdown for the dimension attributes:

| Conformed City Name | Province | Region | Port Flag (0/1) | Facility Type |
| --- | --- | --- | --- | --- |
| **Karachi** | Sindh | South | 1 | Hub / Seaport |
| **Lahore** | Punjab | Central | 0 | Hub / Warehouse |
| **Islamabad** | Islamabad Capital Territory | North | 0 | Office / Hub |
| **Rawalpindi** | Punjab | North | 0 | Depot |
| **Faisalabad** | Punjab | Central | 0 | Hub |
| **Multan** | Punjab | South | 0 | Depot |
| **Peshawar** | Khyber Pakhtunkhwa | North | 0 | Hub |
| **Quetta** | Balochistan | West | 0 | Hub |
| **Sialkot** | Punjab | Central | 0 | Depot |
| **Gujranwala** | Punjab | Central | 0 | Depot |
| **Hyderabad** | Sindh | South | 0 | Depot |
| **Sukkur** | Sindh | North | 0 | Hub |
| **Faisalabad Airport** | Punjab | Central | 0 | Airport Cargo Facility |
| **Gwadar** | Balochistan | South | 1 | Seaport Hub |

---

#### Part 2: Block 2 — The Dimension Structure (`DimCity`)

Every conformed city record is assigned an auto-incrementing integer surrogate key, with the mandatory `-1 Unknown` row reserved at the top.

* **`-1`**: `City_Name = "Unknown"`, `Province = "Unknown"`, `Region = "Unknown"`, `Port_Flag = 0`, `Facility_Type = "Unknown"`
* **`1`**: `City_Name = "Karachi"`, `Province = "Sindh"`, `Region = "South"`, `Port_Flag = 1`, `Facility_Type = "Hub / Seaport"`
* *(...and so on for the remaining conformed cities)*

---

#### Part 3: The Three Judgement Questions

1. **You merged a facility 40 km outside the city into it. Name one business decision that choice quietly made for the company.**
* **Answer:** By merging an out-of-town sorting hub or distant facility into the core city dimension, you quietly made the business decision to **absorb the transit and operational overhead costs of that remote facility into the primary urban delivery zone**, obscuring true last-mile transportation inefficiencies and distorting per-kilometer cost analytics for urban deliveries.


2. **One location is not in the province most people assume. What breaks if you get it wrong, and what breaks if you leave the province blank instead?**
* **Answer:**
* **If you get it wrong:** Provincial performance reporting, tax attribution, and regional KPIs will be structurally misstated, leading executives to misallocate regional budgets and sales incentives.
* **If you leave it blank:** Regional drill-down reports will group those shipments into a massive "Blank/Null" bucket, or worse, cause inner-join drop-offs in star schema queries that filter by province, silently hiding those shipments from regional dashboards.




3. **A new file arrives next month with a spelling nobody has mapped. Walk through exactly what has to happen, in order, for it to appear correctly.**
* **Answer:**
1. **Staging / Exception Catching:** The ETL ingestion pipeline encounters the unmapped string and fails or routes it to an exception log rather than failing silently.
2. **Unknown Fallback:** Because the string isn't in the mapping table, the surrogate key lookup defaults safely to the `-1 Unknown` city key, ensuring no rows are dropped during fact loading.
3. **Conforming Update:** A data engineer or analyst updates the mapping table / data prep script to map the new spelling to its correct conformed city.
4. **Dimension & Fact Refresh:** The ETL job re-runs the dimension loader (if needed) and refreshes/re-processes the fact table so historical and new shipments correctly resolve to the proper `City_SK` instead of `-1`.


### T5 · The Other Four Dimensions Solution

#### Part 1: The Block A Route Trap

* **The Reason for the Mismatch:** A simple automated "Remove Duplicates" on the raw route columns fails to yield the clean expected count because of **hidden data quality anomalies**—specifically trailing/leading whitespaces, inconsistent capitalization (e.g., "Lahore-Karachi" vs. "lahore-karachi"), and legacy aliased route codes embedded in the source data.
* **The Operation That Must Happen First:** Before deduplication or surrogate key generation, you must execute a **string normalization and trimming pass** (converting all text to lowercase/proper case, stripping whitespace, and standardizing delimiter hyphens) so that semantic duplicates are recognized as identical strings.

---

#### Part 2: Dimension Tables Summary (`DimRoute`, `DimMode`, `DimStatus`, `DimDate`)

* **`DimRoute`:** Contains all conformed transit corridors, plus the mandatory `Route_SK = -1` ("Unknown").
* **`DimMode`:** Contains distinct transport modes (e.g., Road, Rail, Air, Sea), plus `Mode_SK = -1` ("Unknown").
* **`DimStatus`:** Contains lifecycle states (e.g., Pending, In Transit, Delivered, Cancelled), plus `Status_SK = -1` ("Unknown").
* **`DimDate`:** Contains a continuous, unbroken calendar span for all 28 days of February 2026 (Day, Weekday/Weekend flag, Month, Year), plus `Date_SK = -1` ("1900-01-01" Unknown date).

---

#### Part 3: Why DimDate Gets 28 Rows When Facts Only Touch 12

* **Answer:** A dimension table models *context*, not just historical activity. Even if PakFreight only operated on 12 distinct days in February 2026, business intelligence reporting requires a **dense calendar continuum**. If you omitted the 16 idle days, time-series visualizations would break, rolling averages would skew, and zero-sales/zero-shipment days could not be rendered accurately on charts or audited properly.

---

#### Part 4: The Four Judgment Questions

1. **Why must dimension tables be built and loaded *before* the fact table?**
* **Answer:** Fact tables depend entirely on the integer surrogate keys (`_SK`) generated by dimension tables. Without dimensions existing first to mint those keys, the fact table loader has nothing valid to look up or foreign-key join against.


2. **What happens in a BI dashboard if a route name changes mid-year and you update it in-place instead of using a Type 2 Slowly Changing Dimension (SCD)?**
* **Answer:** If you overwrite the route name in place, all historical shipments associated with that route will instantly adopt the *new* name, retroactively rewriting history and distorting historical performance metrics for the old route corridor.


3. **Why do we include a `-1 Unknown` row in every single dimension table?**
* **Answer:** It provides a safe fallback for orphaned, missing, or unmapped foreign keys during ETL ingestion. Without it, a single bad record would cause the ETL pipeline to crash or drop data entirely due to referential integrity violations.


4. **Is it acceptable to derive `IsWeekend` using a calculated column in Power BI instead of hardcoding it into `DimDate`?** *(The question with no single right answer)*
* **Answer:** While Power BI DAX can compute `IsWeekend` on the fly, best-practice enterprise dimensional modeling dictates storing static attributes like holidays and weekend flags directly inside the physical `DimDate` table. Storing them pre-calculated ensures query performance optimization, avoids DAX compute overhead at runtime, and allows easy handling of custom regional or corporate calendar definitions that standard formulas might miss.

### T6 · Build FactShipment Solution (15 Marks)

#### 1. Implementation Mechanics (`FactShipment` Structure)

* **Six Integer Keys:** Each row uses a lookup formula (such as `INDEX/MATCH` or `XLOOKUP` wrapped with `IFERROR(..., -1)`) to fetch surrogate keys (`_SK`) from `DimCity`, `DimRoute`, `DimMode`, `DimStatus`, and `DimDate`. Unmatched values safely default to `-1`.
* **Measures:** Additive metrics (e.g., `Weight_kg`, `Revenue_PKR`) carried over directly from `Shipments_Unified`.
* **Degenerate Dimension:** `ShipmentID` remains in the table as a degenerate dimension to uniquely identify each transaction.
* **No Descriptive Text:** All text attributes (city names, route text, status labels) are stripped and left in their respective dimension tables.

---

#### 2. The Three Questions

1. **What `IFERROR(..., -1)` buys you and what it costs you:**
* **Buys:** ETL resilience and referential integrity. Unmapped or newly introduced source strings safely map to `-1` ("Unknown") instead of throwing errors or dropping valuable fact rows.
* **Costs:** It masks data quality failures. Silent fallbacks to `-1` hide missing mapping entries from automated pipeline alerts unless an explicit error-logging audit is run.


2. **Why the text columns do not cross the gap:**
* To prevent data duplication, eliminate update anomalies, and optimize database storage. Storing descriptive text once in a dimension table—rather than repeating it across thousands of fact rows—keeps the fact table lean and performant.


3. **Why `ShipmentID` stays in the fact table even though it joins to nothing:**
* It serves as a **degenerate dimension** (`DD`). While it has no associated dimension table, it retains the source transaction identifier (like a receipt number) for granular drill-through analysis and individual shipment tracking.